# Pya - Tour of AGens

- This notebook contains examples and explanations for available AGens provided
with the pya package. 
- For information about the logic around pya, cf. the
Jupyter notebook pya-examples-agen.ipynb. 

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from pya import Aserver, Asig, startup, device_info
from pya.agen.lib import SinOsc, Line
import pyamapping as pam 
from ipywidgets import interactive

mpl.rcParams['figure.figsize'] = (8,2.5)

device_info();
s = startup() # optionally add device={idx}
(SinOsc(1000) * Line(0.1, 0, 0.1)).play();

## Oscillators

### LFSaw(freq, phase)

This is a Sawtooth Oscillator: 
- non-band-limited, 
- value range is [-1, 1], 
- The frequency (GenOrNum) is given in Hz 
- it starts at zero with positive slope.
- the phase (GenorNum) argument is a normalized phase, i.e. range [0,1] (instead of 0, 2pi)

In [ ]:
from pya.agen.lib import LFSaw
from pya.agen.core import stereo, multi_channel

LFSaw(50, multi_channel(0, 0.25, 0.5, 0.75, 1)).gen_asig(seconds=0.1).plot(offset=3)

In [ ]:
# can be used for audio creation, but mind the aliasing
(LFSaw(50) * Line(0.1, 0, 2)).play()

In [ ]:
# often used for control signal, e.g. here to control a BLImp to create an alert
from pya.agen.lib import BLImp

agfreq = LFSaw(2, 0.5).linlin(-1, 1, 200, 300)
(BLImp(agfreq, 8) * Line(0.1, 0.5, 2)).play()

Here an example where the phase is modulated to detune two LFSaws

In [ ]:
from pya.agen.lib import Env

agenv = Env([0, 0.2, 0.2, 0], [0.5, 3, 0.5])
agsaw = LFSaw(stereo(70, 70), stereo(0, SinOsc(0.4).linlin(-1, 1, 0, 0.25))).mix()
(agsaw * agenv).play()

### LFTri(freq, phase)

This is a Triangle Oscillator: 
- non-band-limited, 
- value range is [-1, 1], 
- The frequency (GenOrNum) is given in Hz 
- it starts at zero with positive slope.
- the phase (GenorNum) argument is a normalized phase, i.e. range [0, 1] (instead of 0, 2pi)
- Technically, a triangle is abs(LFSaw), scaled and shifted to target range

In [ ]:
from pya.agen.lib import LFTri
from pya.agen.core import stereo, multi_channel

LFTri(50, multi_channel(0, 0.25, 0.5, 0.75, 1)).gen_asig(seconds=0.1).plot(offset=3)

- LFTri can be used for audio creation, but mind the aliasing
- The example plays a set of Triangle tones interleaved with SinOsc tones

In [ ]:
for i, f in enumerate([110, 220, 440, 880, 1760, 3520]):
    (LFTri(f, 0) * Line(0.1, 0, 1)).play(onset=i)
    (SinOsc(f, 0) * Line(0.1, 0, 1)).play(onset=i + 0.5)

In [ ]:
# often used for control signal, e.g. here to control a BLImp to create a joddling
from pya.agen.lib import Env

agfreq = (LFTri(5) * Line(0.3, 1, 2)).linlin(-1, 1, 0, 80)
(SinOsc(400, phase=agfreq) * Env([0, 0.2, 0.2, 0], [0.2, 2, 0.2])).play()

- an interesting use of LFTri is as phasor into a buffer, -> AsigRead

### LFPulse(freq, phase, width)

- A non-band-limited rectangular pulse
- value range is [0, 1] (attention, NOT [-1, 1])
- The frequency (GenOrNum) is given in Hz 
- it starts right after the neg-to-pos transition.
- the phase (GenorNum) argument is a normalized phase, i.e. range [0, 1] (instead of 0, 2pi)
- note that zero phase will lead to 1, i.e. the output is perfectly binary.

In [ ]:
from pya.agen.lib import LFPulse


def plot(width=0.2, freq=1000, phase=0.0):
    # clear_output() # NOTICE: I'm calling the global clear_output
    atmp = LFPulse(freq, phase=phase, width=width).gen_asig(seconds=0.002)
    atmp.plot(marker="o", ms=1, lw=0.2, ax=plt.gca())


interactive(plot, width=(0, 1, 0.005), phase=(0, 1, 0.001), freq=(500, 1500, 10))

- modulating the pulse width affects the sharpness of the sound
- be aware of aliasing 

In [ ]:
agfreq = Line(100, 159, 1.0, done="last")
env = Line(0.2, 0, 5)
(LFPulse(agfreq, phase=0, width=LFTri(0.5).linlin(-1, 1, 0, 1)) * env).play();

In [ ]:
# sweep through width
atmp = LFPulse(600, width=Line(0, 0.5, 2.0)).gen_asig()

# plot a spectrogram
atmp.to_stft(nperseg=256).plot(pam.amp_to_db)

# play the output
atmp.gain(0.05).stereo().fade_in(0.01).fade_out(0.01).play(onset=1);

### Pink Noise

A pink noise generator 
- by definition with 1/f power spectral density from ~10 - 24000 Hz.
- implemented by Paul Kellets algorithm
- performenace optimised via numba
- features filter coefficients as state for blockwise computation

In [ ]:
from pya.agen.lib import PinkNoise

apn = PinkNoise().gen_asig(seconds=1)
apn.plot(lw=0.2).gain(0.2).stereo().play(onset=0.5)

In [ ]:
# plot the power spectral density
def psd(asig, nperseg=10000):
    """power spectral density
    estimated using stft as helper for welchs method
    """
    atmp = asig.to_stft(nperseg=nperseg)
    freqs = atmp.freqs[:-1]
    psd = np.mean(np.abs(atmp.stft), axis=1)[:-1]
    return freqs, psd

# plot power spectral density
plt.figure()
nperseg = 2000
plt.plot(*psd(apn, nperseg=nperseg), label="pink noise spectral density")
plt.loglog(); plt.legend(); plt.grid()

# expectation / ground truth = E(f) = 1/f
fmin, fmax = 10, 20000
plt.plot([fmin, fmax], 2/np.sqrt(nperseg) * np.array([1, np.sqrt(fmin/fmax)]), "r-.");

## Systems / Filters

### LeakyIntegrator(gen, coef, yi)

This is an AGen that implements the system y[n] = gen[n] + coef[n] * y[n-1]
- The coefficient coef is GenOrNum.
- yi is the initial filter delay (default: 0.0)

In [ ]:
from pya.agen.lib import LFTri, Env, LeakyIntegrator

ag = Env([0, 0, 1, 0, 0], [0.1] * 4)  # this as 'density'
ag.gen_asig(seconds=1).plot()
LeakyIntegrator(ag / 4410, 1).gen_asig(seconds=1).plot();  # integral = cdf

- perfect integration of LFPulse with width != 0.5 gives a slowly moving ramp
  which can be useful as phasor to scan Buffers

In [ ]:
from pya.agen.lib import LFPulse, LeakyIntegrator

agli = LeakyIntegrator((LFPulse(100, width=0.55) - 0.5) / 44100, coef=1)
agli.gen_asig(seconds=0.1).plot();

- coefficients close to 1, e.g. 0.999 lead to asymptotic (expoential) growth/decay
- until the new input cancels the samplewise loss, here converging to 100

In [ ]:
from pya.agen.lib import Env, OnePole

def plot(q=0.995):
    agenv = Env([1, 4, 2, 2, 5, 5], [0.1, 0, 0.1, 0, 0.1])
    (LeakyIntegrator(agenv, q, yi=1 / (1 - q)) * (1 - q)).gen_asig().plot()
    OnePole(agenv * 1.0, q, yi=1).gen_asig().plot(ls="-.")
    agenv.gen_asig().plot()
    plt.text(0, 4.5, f"q={q:7.4f}")

interactive(plot, q=(0.99, 0.9999, 0.0001))

### Lag und AGen.lag

This is an Agen that implements a OnePole filter to smooth out value changes. 
- Different from OnePole, it is parameterized by tau, the time for a filter to decay to decay_db.
- the default is that tau is the time for reaching the target up to 0.01% (aka -60 dB missing)
- yi is the initial value for the OnePole filter.
- Note that tau is a GenOrNum, so this can be controlled by AGens (see demo below)

In [ ]:
from pya.agen.lib import Lag, LFPulse
Lag(LFPulse(10), tau=0.05, decay_db=-60, yi=1).gen_asig(seconds=1).plot()

In [ ]:
LFPulse(10).lag(0.05).gen_asig(seconds=1).plot()

In [ ]:
LFPulse(10).lag(tau=Line(0.01, 0.3, 1), yi=0).gen_asig(seconds=1).plot()

## Buffer/Asig Processors

- In pya, buffers are stored, manipulated via Asig instances.
- This section presents AGens that read from/write to Asigs
- Asigs can be used for many purposes:
  - as source for waveforms or recorded audio samples
  - to persist audio for later retrieval (e.g. delay loops)
  - for recording audio
  - etc.

### AsigRead 
- This corresponds to BufRd in SuperCollider. 
- Asigs serve as general and flexible kind of multichannel buffer
- The phase argument can be a generator or number (GenOrNum)

In [ ]:
from pya.agen.lib import LFSaw, LFTri, AsigRead, IndexMode

In [ ]:
# load a sound: here 'finger snap'
a1 = Asig("samples/snap.wav")[{0.029:0.25}].norm(0.3).fade_out(0.1)
a1.play().plot(lw=0.5);

The phase argument can be used in different modes, using the IndexMode Enum class
- IndexMode.INDEX: the actual index as the number of samples [0, asig.samples]
- IndexMode.RAD: radian, i.e. `[0, 2pi]`
- IndexMode.ONE: a normalized phase `[0, 1]`
- IndexMode.TIME: time in seconds, i.e. `[0, asig.get_duration()]`

The following demo shows how using a Line and IndexMode.ONE, enables playback 
over a specified duration, i.e. resampling

In [ ]:
t = 0
for dur in [1, 2, 3, 4]:
    ag = AsigRead(a1, Line(0, 1, dur), mode=IndexMode.ONE, loop=True)
    ag.gen_asig(seconds=2).play(onset=t)
    t += 0.8

Using LFSaw as phasor allows to realize a poor-persons time stretching:
- the phasor slowly advances but uses high slope locally

In [ ]:
a2 = Asig("samples/sonification.wav").play()

In [ ]:
dur = a2.get_duration() * 2.0
rate = 40  # try 10, 40, 100
height = 0.14 / rate  # try 0.07, 0.14, 0.28
phasor = LFSaw(rate) * height + Line(height, 1 - height, dur)
ag = AsigRead(a2, phasor, mode=IndexMode.ONE, loop=True)

plt.figure()
phasor.gen_asig().plot(label="phasor")
ag.gen_asig(seconds=5).play().plot(label="audio", lw=0.2)
plt.legend();

- here an interactive version using a slider value as phase offset
- LFTri helps to avoid clicks as scans go forward-backward
- still jumps occur when the position slider is moved


In [ ]:
from ipywidgets import interactive

phase0, lffreq, ratearg = [0], [30], [0.1]
ag = AsigRead(
    asig=a2 * 0.3,
    phase=LFTri(lffreq, 0) / lffreq * ratearg / 8 + phase0,
    mode=IndexMode.ONE,
    loop=True,
).play()

def scrubby(pos=0.2, freq=10, rate=1.0):
    phase0[0], lffreq[0], ratearg[0] = pos, freq, rate

interactive(scrubby, pos=(0, 1, 0.005), freq=(1, 50, 1), rate=(0.1, 1.5, 0.05))

In [ ]:
s.stop()

- here an example to create new timbre by scanning a wave file using modulation
- the perceptual effect can range from vibrato to strange modulation

In [ ]:
# source sound is hitting a glas with a pencil, here played 2 octaves lower
a3 = Asig("samples/ping.mp3").stereo().norm().play(rate=0.25)

In [ ]:
# modulate playback with a sinusoidal phase
agtmp = AsigRead(a3, SinOsc(6).mul(0.0002) + Line(0, 0.4, 2.0), mode=IndexMode.ONE)
agtmp.gen_asig().norm().fade_out(0.2).plot().play()

### LoopAsig(asig, rate, gate, start_pos, start_loop, end_loop, mode, loop)

Generator for playing and looping Asig objects.

- Note that this AGen does not support specifying a custom sample rate as it always
    uses the sample rate of the provided Asig.
- To resample the provided Asig, instead wrap this AGen with a `ResampleGen`.

**Parameters**: 

- asig: The Asig object to play.
- rate: The rate at which to play the Asig: 
  - 1 is normal, -1 is backwards
  - 0.5 is one octave down, 2 is one octave up
- gate: signal to control loop/reset behavior:
  - positive gate starts/resets playback position immediately to start_pos
  - negative gate deactivates loop condition, i.e. plays rest until Asig end.
- start_pos: sample frame to start (and restart on positive gatee)
- start_loop: sample frame to wrap to on a loop condition (i.e. first sample of loop)
- end_loop: sample frame to end loop (i.e. last sample included)
- mode: IndexMode | str (Default: IndexMode.INDEX)
  - either "index" for raw index, "rad" for (0..2pi), "one" for (0..1) range,
    or "time" for time [s].
- loop: boolean, whether the Asig should be looped.

LoopAsig is implemented in close analogy to the SuperCollider UGen LoopBuf.

Example: Loop the 'ca' part of the word sonification, until gate gets 0, then continue until end
- note that by using IndexMode.Time, start_loop and end_loop can be specified in seconds.
- here the gate is a Line (merely for illustration), often a MIDI NoteOff would set gate to 0.

In [ ]:
from pya.agen.lib import LoopAsig, IndexMode, SinOsc, LFPulse
a1 = Asig("samples/sonification.wav")
LoopAsig(a1, rate=0.9, gate=Line(1,-1, 3.2), start_loop=0.60, end_loop=0.7, mode=IndexMode.TIME, loop=False).play();

Example: 
- to show how positive gates (i.e. change from <=0 to positive) retrigger a reset to start_pos, 
- and how negative (i.e. changes from positive to 0 or negative) deactivate the loop criterion.
- for illustration applied to a decaying sind of 200 Hz

In [ ]:
a1 = (SinOsc(200) * Line(1, 0, 0.5, curve=-5)).gen_asig()
agate = LFPulse(4, width=0.7) - Line(0, 2, 1.6)
ag = LoopAsig(a1, rate=1, gate=agate, start_loop=0.05, end_loop=0.06, mode=IndexMode.TIME, loop=False)

agate.gen_asig(seconds=1.2).plot(); plt.grid();
ag.gen_asig().plot(lw=0.3).play();

## Realtime Sensor AGens

- Realtime Sensor AGens are AGens that retrieve sensor data on their generate()
  invocation.
- Examples are MouseX and MouseY.

### MouseX(minval, maxval, mode) and MouseY(minval, maxval, mode)

- read the mouse cursor x (resp y) coordinate as AGen output.

ToDos:
- [ ] should feature a sampling rate for retrieving the cursor coordinates
  - currently positions are queried once per block
- [ ] should interpolate between positions on the sample level
  - currently all block_size samples contain the current value
- [ ] could allow clipping to the current desktop 
  - currently extrapolates beyond [minval, maxval] in multiscreen setups

In [ ]:
from pya.agen.lib import MouseX, MouseY, SinOsc

agfreq = MouseX(200, 400)  # x -> frequency
agvib = MouseY(0, 20)  # y -> vibrato speed
(SinOsc(agfreq) * SinOsc(agvib).linlin(-1, 1, 0, 0.5)).play();

In [ ]:
s.stop()

In [ ]:
from pya.agen.lib import WhiteNoise, BPF

BPF(WhiteNoise(), f_0=MouseX(100, 4000), bw=MouseY(0.1, 2)).lvl(-20).play(rate=1);

In [ ]:
s.stop()

In [ ]:
from pya.agen.lib import MouseButton
# change freq on left click
agm1 = SinOsc(MouseButton(200, 300, 0))
agm1.lvl(-20).play() 
# white noise amp to 0.3 during right click (two fingers)
agm2 = (WhiteNoise() * MouseButton(0, 0.3, button="right")).play(); 
# click left or right mouse button to test

In [ ]:
# parameter control: possible. ToDo: improve interface
agm1.get_nodes()["freq"].offval = 240
agm1.get_nodes()["freq"].onval = 360
# now click the left mouse again...

In [ ]:
# parameters can also be read
agm1.get_nodes()['freq'].offval

In [ ]:
s.stop()

### AudioIn(server, blocking)

- AudioIn is an AGen to process Aserver audio input.
- for it to work, start Aserver with input_flag=True, e.g.
  - pya.startup(input_flag=True)

**Important Notes**:

- The current Aserver pyaudio backend requires the number of channels of input and output device to match
- Audio input is subject to change in future versions
- For Realtime Rendering (playing AGen via Aserver) AudioIn works as intended, because the blocksize of output and input match.
  - on MacOS, default input (e.g. Macbook Microphone) is mono - for that to work 
    with a 2-channel audio, a workaround is to create an Aggregate Audio Device using BlackHole for additional channels.
- If used for Offline Rendering, however, data is merely copied from Aserver, i.e. resulting in blocks used repeatedly.
- The `blocking` flag checks if the AudioIn block number has changed and sleeps until it does.
  - this is a suboptimal behavior and throttles the rendering, but maybe useful in case an Asig is needed.

In [ ]:
from pya import Aserver, startup

# make sure that Aserver is started with input_flag=True, so that input is collected
Aserver.shutdown_default_server()
s = startup(input_flag=True, channels=1)
Aserver.default = s

A basic example, patching AudioIn to output
- Attention: this could result in Feedback loops, best use with headphones
- speak or whistle into the microphone and hear and observe the signal in the scope_gui output...

In [ ]:
from pya.agen.lib import AudioIn, Line
from pya.gui import AGenPlayGUI
AGenPlayGUI()
s.scope_gui()

AudioIn(server=s).mul(0.1).playx()

In [ ]:
s.stop()

Example: get and plot a single buffer of size blocksize (server.bs)
- the reason that gives a single block is that gen_asig would usually try
  to compute 100000 samples - which likely differs from the audio servers blocksize.
  Any data size will be interpreted as a 'stop' by AGen generate() functions.
- execute the following line repeatedly while making some sound

In [ ]:
AudioIn(server=s).gen_asig().norm().plot() 

Example: Using AudioIn in 'offline rendering' (aka gen_asig) of 0.5s 
- Note that Line() is used to implicitly stop rendering at the end of that time
- Note that blocking=True makes AudioIn.generate() block until new data are available
- Note that you can't use gen_asig(seconds=0.5):
  - that would override / disable the block_size argument and due to the block_size
    mismatch only a single block would result

In [ ]:
ag = AudioIn(server=s, blocking=True) * Line(1, 0, 0.5)
ag.gen_asig(block_size=s.bs).norm().plot()

## Envelopes and Control Signals

### Release()

Release is an AGen for gated fading-out on gate <= 0.

- Release starts with output value 1
- on gate <= 0, Release starts a release curve,
  starting from value 1, ending at 0 after specified duration.
- at the end, the AGen behaves according to its done argument, the default is 'stop'.
- use curve to control the shape of the release, curve currently determines
  the order for a polynomial mapping [warning: this could change in future versions].

Parameters:
- gate: The gate signal (default 1), release starts once gate drops <= 0
- duration: The duration of the fade out in seconds.
- curve: The curve of the fade out (1=linear, $f_c(t) = (1-t/dur)^c$

Release can be used to stop an AGen on a signal, e.g. 
- using a MIDI NoteOn callback to create/play an AGen
- using MIDI NoteOff callback to set its gate to 0

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pya.agen.lib import Release, Line

agate = Line(1,-1, 1)
agate.gen_asig().plot(label="gate signal")
ag = Release(gate=agate, duration=1.5, curve=1.5)
ag.gen_asig(block_size=1000).plot(label="Release output");
plt.grid(); plt.legend();

Example: (Realtime Control) Play a tone indefinitely until gate is set to zero (following code cell)

In [ ]:
gate = [1]
(SinOsc(800) * Release(gate, duration=1.0, curve=3)).mul(0.1).dup(2).play();

In [ ]:
gate[0] = 0

## AGen service functions and implicit AGens

The following AGen functions are introduced to make AGen writing more consice.

### scaling via `.mul()` and `.lvl()`, shifting via `.add()`

- Often AGens require scaling, yet mul is not a standard part of AGens as it is in Supercollider.
- According to the Zen of Python 
  >There should be one-- and preferably only one --obvious way to do it.
  and scaling is best done by the multiplication '*'.
- However, on writing code, e.g. `SinOsc(330)*0.2` stops the flow, as one cannot
  simply append `.play()` of whatever is needed. Instead one has to wrap the
  product in parenthesis, making code less readable. 
- `.mul()` solves this - the argument is a scalar factur
- `.lvl()` allows to provide a deciBel argument scaling correspondingly
- `.add()` adds an offset without breaking the daisy chain logic

In [ ]:
SinOsc(220).mul(0.2).gen_asig(1000).plot(label='scaled by 0.2')
SinOsc(220).lvl(-20).gen_asig(1000).plot(label='scaled by -20 dB')
SinOsc(880).lvl(-40).add(0.15).gen_asig(1000).plot(label='shifted by 0.1')
plt.legend(); plt.grid()

### duplicate mono AGens via `.dup()`

Multichannel AGens can be created by using MultiChannelGen().
- often a on-the-fly expansion is needed and encapsulating an AGen would create
  unnecessary hassle.
- here `.dup(n)` allows to promote any AGen into an array with n times many 
  channels.
- A frequent use case is to turn a monophonic AGen into stereo prior to play().

In [ ]:
(SinOsc(400) * Line(1,0,1.5)).dup(2).play();

In [ ]:
# multi-channel AGens are also duplicated as expected
from pya.agen.core import stereo
SinOsc(stereo(200,400)).mul(0.3).dup(4).gen_asig(seconds=0.1).plot(offset=1);